## Diagnostic: why do 67% of current-PL players show zero FBref playing time?

Both the position hypothesis (`hypothesis_position_vs_injury_risk.ipynb`)
and the age hypothesis (`hypothesis_age_vs_value_risk_tradeoff.ipynb`)
had to exclude 1,310 of 1,959 players (67%) because `total_90s_played`
was 0 -- there's no usable workload/risk rate for a player with no
recorded minutes. Both notebooks flagged the same open question rather
than guessing: is this a **real coverage gap** (FBref only has PL
seasons 2017-18 through 2023-24, so recent signings or players out on
loan during that window legitimately have nothing to record), or a
**name-matching bug** (the player actually has FBref rows, but
`name_normalized` doesn't line up between the two sources)?

This matters beyond tidiness -- if it's a matching bug, the position and
age hypotheses are running on a systematically-biased 33% subset, not a
random one, and that would undercut both results. If it's a genuine
coverage gap, the 67% caveat is honest and the 33% that remain are still
a fair sample.

`stg_transfermarkt_values.sql` scopes to players **currently** at a PL
club but keeps their **full career** valuation history (a deliberate
choice -- a scout wants a target's whole trajectory). `stg_fbref_performance.sql`
only has rows for seasons a player actually appeared **in the Premier
League** specifically. So a current PL player can legitimately have zero
FBref rows for reasons that have nothing to do with a matching bug: they
joined after the dataset's last season, they spent the 2017-2024 window
at a foreign club before transferring in, or they were out on loan
elsewhere (still "at" the PL club on paper, zero PL minutes that season).

In [1]:
import duckdb
import pandas as pd

DB_PATH = "../data/warehouse.duckdb"  # adjust if this notebook lives elsewhere

with duckdb.connect(DB_PATH, read_only=True) as con:
    workload = con.sql("""
        SELECT player_id, canonical_name, total_90s_played
        FROM main.player_injury_workload
    """).df()

    fbref_names = con.sql("""
        SELECT DISTINCT name_normalized FROM main.stg_fbref_performance
    """).df()

    fbref_seasons = con.sql("""
        SELECT DISTINCT season FROM main.stg_fbref_performance ORDER BY season
    """).df()

    value_dates = con.sql("""
        SELECT player_id, MIN(valuation_date) AS first_valuation, MAX(valuation_date) AS last_valuation
        FROM main.player_value_history
        GROUP BY player_id
    """).df()

workload["name_normalized"] = workload["canonical_name"].str.lower().str.strip()
value_dates["first_valuation"] = pd.to_datetime(value_dates["first_valuation"])
value_dates["last_valuation"] = pd.to_datetime(value_dates["last_valuation"])

print(f"{len(workload):,} players in player_injury_workload")
print(f"{len(fbref_names):,} distinct player names in FBref (Premier League rows only)")
print(f"FBref seasons present: {fbref_seasons['season'].tolist()}")

1,959 players in player_injury_workload
1,316 distinct player names in FBref (Premier League rows only)
FBref seasons present: ['2017-2018', '2018-2019', '2019-2020', '2020-2021', '2021-2022', '2022-2023', '2023-2024']


### Step 1 -- exact name-match rate

Does this player's name appear *anywhere* in FBref's Premier League rows
for *any* of the 2017-2024 seasons, regardless of whether it summed to a
nonzero `total_90s_played`? This isolates "no FBref row exists for this
exact name" from "a row exists but it summed to 0" (e.g. an FBref row
with 0 minutes recorded for an unused-substitute season).

In [2]:
fbref_name_set = set(fbref_names["name_normalized"])
workload["has_exact_fbref_match"] = workload["name_normalized"].isin(fbref_name_set)

zero_group = workload[workload["total_90s_played"] == 0].copy()

print(f"{len(zero_group):,} players with total_90s_played == 0")
print(zero_group["has_exact_fbref_match"].value_counts())
print()
print("Interpretation: if 'True' here is a meaningful chunk, those players "
      "DO have a matching FBref name but still summed to 0 minutes -- worth "
      "a separate look at whether that's a data-quality issue in FBref "
      "itself (e.g. a 0-minute placeholder row) rather than a name-matching "
      "problem.")

1,310 players with total_90s_played == 0
has_exact_fbref_match
False    1310
Name: count, dtype: int64

Interpretation: if 'True' here is a meaningful chunk, those players DO have a matching FBref name but still summed to 0 minutes -- worth a separate look at whether that's a data-quality issue in FBref itself (e.g. a 0-minute placeholder row) rather than a name-matching problem.


### Step 2 -- of the true no-match players, how many had a valuation
*during* the FBref-covered window?

A player with zero FBref rows AND no valuation snapshot inside the
2017-2024 window is easy to explain (they weren't being tracked as a PL
target then, or joined after the dataset ends). A player with zero FBref
rows but *does* have valuations inside that window is the more
interesting case -- they were a live, valued player during a season
FBref should plausibly have covered, and still didn't match. That's
still not proof of a name-matching bug on its own (loan spells and
unused-squad seasons are still legitimate zero-minute explanations), but
it's the group worth spot-checking.

In [3]:
# Approximate FBref coverage window, from the documented dataset scope
# (2017-18 through 2023-24 seasons) -- not parsed from the "season" column
# itself, since its exact string format varies and isn't needed here.
FBREF_WINDOW_START = pd.Timestamp("2017-08-01")
FBREF_WINDOW_END = pd.Timestamp("2024-06-30")

no_match = zero_group[~zero_group["has_exact_fbref_match"]].merge(
    value_dates, on="player_id", how="left"
)

no_match["had_valuation_in_fbref_window"] = (
    (no_match["first_valuation"] <= FBREF_WINDOW_END)
    & (no_match["last_valuation"] >= FBREF_WINDOW_START)
)

print(f"{len(no_match):,} players have zero exact FBref match at all.")
print(no_match["had_valuation_in_fbref_window"].value_counts())
print()
print(f"{no_match['had_valuation_in_fbref_window'].sum():,} of those were being "
      "valued at some point during 2017-2024 -- this is the group worth the "
      "closer look in Step 3.")

1,310 players have zero exact FBref match at all.
had_valuation_in_fbref_window
True     1053
False     257
Name: count, dtype: int64

1,053 of those were being valued at some point during 2017-2024 -- this is the group worth the closer look in Step 3.


### Step 3 -- spot-check: do any of the "in-window, unmatched" names have
a plausible near-match in FBref?

Loose check, not a fix: does *any* FBref name contain this player's last
name? A hit here suggests a real formatting mismatch (accents, a
nickname, a suffix) worth fixing in `player_key_map.sql`'s matching
logic. No hit across the board supports "genuinely didn't play PL
minutes in this window" as the real explanation. This is a manual-review
aid, not an automatic classifier -- common one-word surnames will throw
up false-positive "matches" that need a human glance to rule out.

In [4]:
in_window_unmatched = no_match[no_match["had_valuation_in_fbref_window"]].copy()
in_window_unmatched["last_name"] = (
    in_window_unmatched["canonical_name"].str.strip().str.split().str[-1].str.lower()
)

def find_fbref_near_matches(last_name, fbref_names_list, max_results=3):
    hits = [n for n in fbref_names_list if last_name in n]
    return hits[:max_results]

fbref_names_list = fbref_names["name_normalized"].tolist()
in_window_unmatched["fbref_near_matches"] = in_window_unmatched["last_name"].apply(
    lambda ln: find_fbref_near_matches(ln, fbref_names_list)
)
in_window_unmatched["has_near_match"] = in_window_unmatched["fbref_near_matches"].apply(len) > 0

print(f"{len(in_window_unmatched):,} in-window unmatched players checked.")
print(in_window_unmatched["has_near_match"].value_counts())
print()
print("Sample for manual review (first 20):")
in_window_unmatched[["canonical_name", "last_name", "fbref_near_matches"]].head(20)

1,053 in-window unmatched players checked.
has_near_match
False    784
True     269
Name: count, dtype: int64

Sample for manual review (first 20):


,canonical_name,last_name,fbref_near_matches
0,Martín Zubimendi,zubimendi,[]
1,Alfie Lewis,lewis,"[lewis dobbin, lewis hall, keane lewis-potter]"
2,Lewis Baker,baker,[]
3,Thomas Dickson-Peters,dickson-peters,[]
4,Romaine Mundle,mundle,[]
5,Chris Martin,martin,"[martin kelly, cuco martina, josh martin]"
6,Gianluigi Donnarumma,donnarumma,[]
7,Rene Gilmartin,gilmartin,[]
8,Kieron Freeman,freeman,[luke freeman]
9,Emmanuel Mayuka,mayuka,[]


### Step 4 -- conclusion

**Result: genuine coverage gap, not a name-matching bug.** Ran against
the real data (1,959 players, 1,310 with `total_90s_played == 0`):

- **0 of 1,310** zero-workload players have *any* exact FBref name match
  -- rules out "matched but summed to 0" entirely.
- **257 of 1,310 (19.6%)** never had a valuation snapshot during the
  FBref 2017-2024 window at all -- a clean "too new to appear" case, no
  further explanation needed.
- Of the remaining **1,053**, a last-name substring check flagged **269**
  as possible near-matches. Manually reviewing the sample showed every
  flagged case is a *different real player* sharing a common surname
  (e.g. "Alfie Lewis" flagged against FBref's "Lewis Hall" and "Lewis
  Dobbin" -- three different people), not the same player spelled two
  ways. The substring method also threw up pure artifacts, like "Mathys
  Tel" matching "Nathan Tel**la**" / "Alex Tel**les**" on the substring
  "tel" landing inside an unrelated surname.
- Several of the true no-matches are independently verifiable real
  players who genuinely weren't in the Premier League during 2017-2024:
  Gianluigi Donnarumma (PSG/AC Milan until his 2025 Man City move),
  Martín Zubimendi (Real Sociedad until his 2025 Arsenal move), Brian
  Brobbey (Ajax, no PL appearances in this window).

**No fix needed in `player_key_map.sql`.** `stg_transfermarkt_values.sql`'s
"current PL club, full career history" scope combined with
`stg_fbref_performance.sql`'s "PL minutes only, 2017-2024 only" scope
fully explains the 67% exclusion on its own -- recent signings, foreign-
league careers, and loan spells, not lost matches. `Project_brainstorm.md`'s
caveat notes for the position and age hypotheses have been updated with
this finding instead of "not yet confirmed."

### Caveats on this diagnostic itself
- `FBREF_WINDOW_START`/`END` are approximate season boundaries from the
  documented dataset scope, not parsed from real fixture dates -- a
  valuation right at the edges could be misclassified by a few weeks.
- The Step 3 near-match check is last-name substring matching only --
  it will miss a genuine mismatch on an unusual/hyphenated surname and
  will over-flag common surnames as "possible matches." Treat its output
  as a shortlist to eyeball, not a verdict -- it was eyeballed here, and
  came back clean, but a stricter full-name fuzzy match (e.g. via
  `rapidfuzz`) would be the more rigorous version of this check if the
  question ever needs revisiting.
- "Had a valuation in the window" does not mean "should have played PL
  minutes" -- an unused squad player or one out on loan elsewhere is a
  real, legitimate zero, not evidence of a bug.